# 00 — Bronze Inventory

**Objetivo:** establecer qué snapshots existen realmente en Bronze, cuánto pesan y cuántas filas/columnas contienen.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Preguntas
- ¿Están presentes exactamente las siete fuentes esperadas?
- ¿Cuál es el tamaño real de cada snapshot local?
- ¿Cuántas filas y columnas tiene cada CSV descargado?
- ¿Alguna fuente necesita fallback de parsing?

**Guardrail:** `data/bronze` es inmutable. Este notebook solo lee.

In [ ]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

In [ ]:
from argos.profiling.inventory import build_inventory
from argos.common.reports import write_report

profiling_cfg = PROJECT_CONFIG["profiling"]

inventory = build_inventory(
    FILES,
    DATASETS,
    sample_size=profiling_cfg["csv_sample_size"],
    compute_sha256=profiling_cfg["compute_sha256"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

report_path = write_report(
    inventory,
    paths.reports / "profiling" / "dataset_inventory.parquet",
)

inventory[[
    "dataset", "source_id", "filename", "file_size_gb",
    "row_count", "column_count", "parse_mode", "elapsed_seconds"
]]

## Criterio de salida
No continúes si falta una fuente, si hay dos snapshots activos de la misma fuente o si un archivo no puede parsearse de manera reproducible.

**Decisión a documentar:** snapshot de Bronze que será la base de M1.